### Cell 1: 初始化与配置

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# 引入自定义模块
from kg_pipeline import KGPipeline
from df_helpers import ResolveCoreferences, ExtractConcepts, df2Graph, graph2Df, contextual_proximity

# --- 配置 ---
# 请根据实际路径修改 INPUT_FILE
INPUT_FILE = "../data_input/dataset/hotpot/hotpot_dev_distractor_v1.json"
OUTPUT_DIR = "../data_output/dataset/hotpot/ds1000_2" # 建议修改输出目录，避免覆盖之前的结果

# [修改点 1] 设置范围
START_INDEX = 0  # 起始位置
END_INDEX = 1000    # 结束位置 (None 表示到最后)
# 是否强制重新生成 (True: 忽略缓存重新运行; False: 优先加载缓存)
REGENERATE = False  

# 初始化 Pipeline 实例
pipeline = KGPipeline(output_dir=OUTPUT_DIR)

d:\Anaconda_envs\envs\andelie\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1 - 数据加载与分块 (Chunking)

In [2]:
# 将原始 JSON 数据加载并分割为 Chunks
if REGENERATE:
    df_chunks = pipeline.load_and_split_data(
        json_path=INPUT_FILE, 
        start_index=START_INDEX, # 新增参数
        end_index=END_INDEX
    )
else:
    # 尝试加载缓存
    chunk_cache = Path(OUTPUT_DIR) / "chunk.csv"
    if chunk_cache.exists():
        df_chunks = pd.read_csv(chunk_cache, sep="|")
    else:
        # 如果缓存不存在，强制重新生成
        df_chunks = pipeline.load_and_split_data(json_path=INPUT_FILE, start_index=START_INDEX, end_index=END_INDEX)

print(f"Loaded {len(df_chunks)} chunks.")
print(df_chunks.head(2))

Loaded 9958 chunks.
   context_id  chunk_id             title  \
0           0         0    Ed Wood (film)   
1           0         1  Scott Derrickson   

                                                text  
0  Ed Wood (film) information: Ed Wood is a 1994 ...  
1  Scott Derrickson information: Scott Derrickson...  


## Step 2 - 指代消解 (Coreference Resolution)

In [3]:
# 对 Chunks 进行指代消解
# REGENERATE = True
resolved_path = Path(OUTPUT_DIR) / "resolved_chunks.csv"

if REGENERATE or not resolved_path.exists():
    df_chunks = ResolveCoreferences(df_chunks,max_workers=50)
    df_chunks.to_csv(resolved_path, sep="|", index=False)
else:
    print("Loading resolved chunks from cache...")
    df_chunks = pd.read_csv(resolved_path, sep="|")

print(df_chunks[['text', 'resolved_text']].head(2))

Loading resolved chunks from cache...
                                                text  \
0  Ed Wood (film) information: Ed Wood is a 1994 ...   
1  Scott Derrickson information: Scott Derrickson...   

                                       resolved_text  
0  Ed Wood (film) information: Ed Wood is a 1994 ...  
1  Scott Derrickson information: Scott Derrickson...  


## Step 3 - 生成 Chunk 嵌入向量 (Embeddings)

In [4]:
# REGENERATE = True
# 定义文件名
final_emb_filename = "chunks_with_embeddings.parquet"
temp_title_filename = "temp_title_embeddings.parquet" # 临时文件
final_path = Path(OUTPUT_DIR) / final_emb_filename
temp_path = Path(OUTPUT_DIR) / temp_title_filename

# 1. 只有 REGENERATE 为 True 时才重新计算，否则直接加载
if REGENERATE or not final_path.exists():
    print("🚀 [Notebook] 开始双重向量计算 (Content + Title)...")

    # --- 第一步：计算正文 (Content) 向量 ---
    # 这会生成 'embedding' 列
    df_content = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='text', 
        id_col='chunk_id', 
        file_name=final_emb_filename 
    )

    # --- 第二步：计算标题 (Title) 向量 ---
    # 为了不覆盖原来的文件，我们先存到一个临时文件里
    # 注意：pipeline 默认会把列名存为 'embedding'
    df_title = pipeline.compute_embeddings(
        df=df_chunks, 
        text_col='title', 
        id_col='chunk_id', 
        file_name=temp_title_filename 
    )

    # --- 第三步：在 Notebook 中执行合并与重命名 ---
    print("🔄 [Notebook] 正在合并 Title 向量...")
    
    # 提取 df_title 中的 chunk_id 和 embedding，并将 embedding 改名为 title_embedding
    # 注意：这里我们只取需要的两列，防止其他列重复
    df_title_subset = df_title[['chunk_id', 'embedding']].copy()
    df_title_subset.rename(columns={'embedding': 'title_embedding'}, inplace=True)
    
    # 将标题向量 merge 到主数据中
    df_final = pd.merge(df_content, df_title_subset, on='chunk_id', how='left')
    
    # --- 第四步：手动保存合并后的结果 ---
    # 必须把 numpy array 转回 list 才能存 Parquet (这是 Pipeline 内部做的事，我们在外面手动做一遍)
    for col in ['embedding', 'title_embedding']:
        if col in df_final.columns:
            # 检查第一行如果是 numpy 数组，则执行转换
            if not df_final[col].empty and isinstance(df_final[col].iloc[0], np.ndarray):
                df_final[col] = df_final[col].apply(lambda x: x.tolist())

    df_final.to_parquet(final_path, index=False)
    
    # 清理临时文件
    if temp_path.exists():
        os.remove(temp_path)
        
    print(f"✅ [Notebook] 双重向量合并完成！已保存至 {final_path}")
    df_chunks = df_final # 更新当前内存中的 df_chunks

else:
    print(f"🔍 [Notebook] 加载现有缓存: {final_path}")
    df_chunks = pd.read_parquet(final_path)

# --- 验证数据 ---
print("-" * 30)
first_row = df_chunks.iloc[0]
print(f"Data Loaded. Columns: {df_chunks.columns.tolist()}")
if 'embedding' in df_chunks.columns:
    print(f"Content Vector Dim: {len(first_row['embedding'])}")
if 'title_embedding' in df_chunks.columns:
    print(f"Title Vector Dim:   {len(first_row['title_embedding'])}")

🔍 [Notebook] 加载现有缓存: ..\data_output\dataset\hotpot\ds1000_2\chunks_with_embeddings.parquet
------------------------------
Data Loaded. Columns: ['context_id', 'chunk_id', 'title', 'text', 'resolved_text', 'embedding', 'title_embedding']
Content Vector Dim: 1024
Title Vector Dim:   1024


## Step 4 - 概念/实体抽取 (Concept Extraction)

In [ ]:
REGENERATE = True
# 并行从文本中抽取实体
concepts_path = Path(OUTPUT_DIR) / "extracted_concepts.csv"

if REGENERATE or not concepts_path.exists():
    df_concepts = ExtractConcepts(dataframe=df_chunks, max_workers=30)
    df_concepts.to_csv(concepts_path, sep="|", index=False)
else:
    df_concepts = pd.read_csv(concepts_path, sep="|")

print(f"Extracted {len(df_concepts)} concepts.")

--- 步骤: 开始并行抽取 9958 个 Chunk 的概念节点 ---
🚀 正在提交 9958 个任务到线程池...
⏳ 开始并行处理...


 10%|█         | 1002/9958 [14:35<1:23:14,  1.79chunk/s]

## Step 5 - 实体标准化与对齐 (Entity Standardization)

In [ ]:
# 这一步包含了最复杂的逻辑：
# 1. 聚合实体同义词 (Rich Text)
# 2. 计算实体 Embedding
# 3. 使用 HAC + Genealogy Penalty对齐
# 4. 生成标准实体名称
# 这一步包含了最复杂的逻辑：聚合同义词 -> 计算Embedding -> HAC + Genealogy Penalty -> 生成标准名称
# REGENERATE = False

std_path = Path(OUTPUT_DIR) / "dp_extracted_concepts.csv"
if REGENERATE or not std_path.exists():
    # 执行标准化流程 (内部会保存文件)
    df_concepts_std = pipeline.standardize_entities(df_concepts)
else:
    print(f"🔍 加载缓存的标准化实体: {std_path}")
    df_concepts_std = pd.read_csv(std_path, sep="|")

print("Standardization sample:")
print(df_concepts_std[['Entity', 'Standard_Entity', 'cluster_id']].head())

🔍 加载缓存的标准化实体: ..\data_output\dataset\hotpot\ds1000\dp_extracted_concepts.csv
Standardization sample:
          Entity Standard_Entity  cluster_id
0        Ed Wood         Ed Wood         115
1     Lisa Marie      Lisa Marie          43
2  Martin Landau   Martin Landau          41
3        Ed Wood     Ed Wood Sr.          25
4    Bela Lugosi     Bela Lugosi          64


## Step 6 - QA 数据提取

In [ ]:
# 从原始数据提取问题和答案对
qa_path = Path(OUTPUT_DIR) / "qa.csv"


if REGENERATE or not qa_path.exists():
    df_qa = pipeline.extract_qa_pairs(INPUT_FILE, max_contexts=(END_INDEX-START_INDEX))
else:
    print(f"🔍 加载缓存的 QA 数据: {qa_path}")
    df_qa = pd.read_csv(qa_path, sep="|")

print(df_qa.head(2))

🔍 加载缓存的 QA 数据: ..\data_output\dataset\hotpot\ds1000\qa.csv
                                            question             answer  \
0  Were Scott Derrickson and Ed Wood of the same ...                yes   
1  What government position was held by the woman...  Chief of Protocol   

   context_id  
0           0  
1           1  


### Step 7 - 基础图谱构建 (Relation Extraction)

In [ ]:
# 1. 准备实体映射字典 {chunk: {original_name: standard_name}}
entity_map = pipeline.generate_entity_map_for_graph(df_concepts_std)

# REGENERATE = False

# 2. 执行关系抽取
graph_path = Path(OUTPUT_DIR) / "graph.csv"

if REGENERATE or not graph_path.exists():
    # 使用 LLM 抽取关系，传入 entity_map 以实现引导式抽取
    relations_list = df2Graph(df_chunks, entity_map,max_workers=30)
    df_graph = graph2Df(relations_list)
    df_graph.to_csv(graph_path, sep="|", index=False)
else:
    df_graph = pd.read_csv(graph_path, sep="|")

print(f"Extracted {len(df_graph)} relations.")
print(df_graph.head())

d:\Code\jupyter\knowledge_graph\hotpot\kg_pipeline.py:387: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return valid.groupby(['context_id', 'chunk_id']).apply(


--- 步骤: 开始并行抽取图谱关系 ---
🚀 正在提交 10004 个任务到线程池...
⏳ 开始并行处理...


 81%|████████  | 8096/10004 [1:57:19<18:34,  1.71chunk/s]  


🛑 ID 4497 最终失败: Request timed out....


 89%|████████▉ | 8914/10004 [2:08:50<09:34,  1.90chunk/s]  

⚠️ 整体解析失败，尝试逐个提取对象...
✅ 鲁棒提取成功恢复 8 条记录。


100%|██████████| 10004/10004 [2:25:20<00:00,  1.15chunk/s]



🧹 正在清理线程池（丢弃卡死的线程）...
✅ 处理结束。成功获取结果: 10003/10004
✅ 关系抽取完成，共提取 111297 条边。
Extracted 111297 relations.
   context_id         node_1       node_2  \
0           0    Ed Wood Sr.         1994   
1           0  Martin Landau  Bela Lugosi   
2           0    Ed Wood Sr.  Bela Lugosi   
3           0    Johnny Depp  Ed Wood Sr.   
4           0     Tim Burton  Ed Wood Sr.   

                                                edge  chunk_id  
0  The film 'Ed Wood', which is about Ed Wood Sr....         0  
1  Martin Landau played the role of Bela Lugosi i...         0  
2  The film 'Ed Wood' explores Ed Wood Sr.'s rela...         0  
3  Johnny Depp starred as Ed Wood Sr. in the film...         0  
4  Tim Burton directed and produced the biographi...         0  


### Step 8 - 计算上下文共现关系 (Contextual Proximity)

In [ ]:
# 基于 Chunk 共现计算额外的图谱边
# REGENERATE = True

df_proximity = contextual_proximity(df_graph)
df_proximity.to_csv(Path(OUTPUT_DIR) / "contextual_proximity.csv", sep="|", index=False)

print(f"Generated {len(df_proximity)} proximity edges.")
print(df_proximity.head())

Generated 254096 proximity edges.
   context_id                node_1                   node_2 chunk_id  count  \
7         320      " I Like My Job"             Duke Tumatoe     3255     46   
38        638      "1 Night 2 Days"             Lee Seung-gi     6427     22   
50          6  "100 Latinos Madrid"       Amaruk Kayshapanta       71     18   
66        314                  "19"  "Critics' Choice" award     3194      6   
67        314                  "19"         "Hometown Glory"     3194     12   

                    edge  
7   contextual_proximity  
38  contextual_proximity  
50  contextual_proximity  
66  contextual_proximity  
67  contextual_proximity  


## Step 9 - Merge Entities & Compute Dual Embeddings

In [ ]:
# --- New Step: Merge and Enrich ---
# 聚合实体信息并计算向量 (vec_entity 和 vec_desc)
merged_path = Path(OUTPUT_DIR) / "concepts_merged_with_vectors.parquet"
REGENERATE = True
# 1. 如果强制重新生成，先删除缓存文件
if REGENERATE and merged_path.exists():
    print(f"🗑️ [REGENERATE] 删除旧聚合缓存: {merged_path}")
    os.remove(merged_path)

# 2. 调用 Pipeline (内部逻辑: 无文件则计算并保存，有文件则加载)
df_merged_vectors = pipeline.merge_and_embed_concepts(df_concepts=df_concepts_std)

print("Merged Data Sample:")
print(df_merged_vectors[['Standard_Entity', 'description', 'vec_entity']].head(2))

🗑️ [REGENERATE] 删除旧聚合缓存: ..\data_output\dataset\hotpot\ds1000\concepts_merged_with_vectors.parquet
🚀 [Pipeline] 开始实体聚合...
🔄 [Helper] 正在聚合实体数据...
✅ [Helper] 聚合完成。原始行数: 125497 -> 聚合后行数: 104057
🛠️ [Pipeline] 构建增强实体文本...
🚀 [Pipeline] 计算 Enriched Entity Vectors...


Vec: Entity: 100%|██████████| 1626/1626 [26:40<00:00,  1.02it/s]


🚀 [Pipeline] 计算 Description Vectors...


Vec: Desc: 100%|██████████| 1626/1626 [27:01<00:00,  1.00it/s]


💾 [Pipeline] 保存聚合向量表到: ..\data_output\dataset\hotpot\ds1000\concepts_merged_with_vectors.parquet
Merged Data Sample:
        Standard_Entity                                     description  \
0            Tim Burton  The director and producer of the film Ed Wood.   
1  Sarah Jessica Parker    A member of the supporting cast of the film.   

                                          vec_entity  
0  [-0.011303096, 0.007036083, 0.026462099, 0.027...  
1  [0.004416721, -0.0030213601, -0.021587733, 0.0...  
